In [1]:
import logging
from pulao.logging import init_logging

import pandas as pd

from IPython.display import display

from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value, datetime=datetime, turnover=money, open_price=open, close_price=close, high_price=high, low_price=low, volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#

# region . Plotly - Cbar
#
df_cbar = cbar_manager.df_cbar.to_pandas()
df_cbar["datetime"] = pd.date_range("2025-01-01", periods=df_cbar.shape[0], freq="h")
df_cbar["open_price"] = df_cbar["high_price"]
df_cbar["close_price"] = df_cbar["low_price"]

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")

for i, row in enumerate(df_swing.itertuples()):
    df = df_cbar[(df_cbar["id"] >= row.start_id) & (df_cbar["id"] <= row.end_id)]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs = []
ys = []
texts = []
text_positions  = []
for i, row in enumerate(df_swing.itertuples()):
    xs += [row.start_datetime, row.end_datetime, None]
    if row.direction == SwingDirection.UP:
        start_price = df_cbar[(df_cbar["id"] == row.start_id)]["low_price"]
        end_price = df_cbar[(df_cbar["id"] == row.end_id)]["high_price"]
        text_positions += ['bottom center', 'top center', "middle center"]
    else:
        start_price = df_cbar[(df_cbar["id"] == row.start_id)]["high_price"]
        end_price = df_cbar[(df_cbar["id"] == row.end_id)]["low_price"]
        text_positions += ['top center', 'bottom center', "middle center"]
    ys += [start_price, end_price, None]
    texts += [row.start_id, row.end_id, None]

# 添加index列
df_cbar.insert(0, "index", range(len(df_cbar)))
df_swing.insert(0, "index", range(len(df_swing)))
# endregion

fig = make_subplots(
    rows=1, cols=1
)
fig.add_trace(go.Candlestick(
    x=df_cbar['datetime'],
    open=df_cbar['open_price'],
    high=df_cbar['high_price'],
    low=df_cbar['low_price'],
    close=df_cbar['close_price'],
    name='K线',
), row=1, col=1)

df_fractal_bottom = df_cbar[(df_cbar['fractal_type'] == FractalType.BOTTOM)]

fig.add_trace(go.Scatter(
    x=df_fractal_bottom['datetime'],
    y=df_fractal_bottom['low_price'] * 0.995,   # 放在K线最低价下方一点，不遮挡蜡烛
    mode='markers+text',
    name='swing point low',
    marker=dict(
        symbol='triangle-up',
        size=14,
        color='#1E90FF',
    ),
    text=df_fractal_bottom["index"],      # ← 添加序号
    textposition="bottom center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段低点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)


df_fractal_top = df_cbar[(df_cbar['fractal_type'] == FractalType.TOP)]

fig.add_trace(go.Scatter(
    x=df_fractal_top['datetime'],
    y=df_fractal_top['high_price'] * 1.005,  # 放在K线最高价上方一点
    mode='markers+text',
    name='swing point high',
    marker=dict(
        symbol='triangle-down',
        size=14,
        color='#FF4500',
    ),
    text=df_fractal_top["index"],      # ← 添加序号
    textposition="top center",              # ← 文字位置
    textfont=dict(size=10, color="white"),
    hovertemplate=
        "<b>波段高点</b><br>" +
        "时间: %{x}<br>" +
        "价格: %{y:.2f}<br>" +
        "<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=xs,
    y=ys,
    name='swing segments',
    mode='lines+text',    # lines+text 支持显示文字
    line=dict(width=2, color='orange'),
    text=texts,
    textposition=text_positions,  # 文字位置
))

# title = 'CBar Chart - K线合并处理'
title = 'CBar Chart - 分形重叠处理'
# title = 'CBar Chart - 次级别处理'
# title = 'CBar Chart - 连续同级别同向高低点处理'
fig.update_layout(title=title,
                  height=900,
                  hovermode='x unified',  # X轴悬停联动虚线
                  )

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)

fig.show()

# endregion

{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-26 21:57:24"}
{"cbar_count": 1, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-26 21:57:24"}
{"cbar_count": 2, "event": "用于组成分形的cbar数量不够", "level": "warning", "logger": "pulao.swing.swing_manager", "time": "2025-11-26 21:57:24"}
{"fractal": "Fractal(left=CBar(id=119557145480921088, start_id=119557145476726784, end_id=119557145480921088, high_price=942.2349853515625, low_price=935.9240112304688, created_at=datetime.datetime(2025, 11, 26, 21, 57, 24, 747000), fractal_type=0), middle=CBar(id=119557145485115392, start_id=119557145485115392, end_id=119557145485115392, high_price=939.052001953125, low_price=930.2979736328125, created_at=datetime.datetime(2025, 11, 26, 21, 57, 24, 748000), fractal_type=-1), right=CBar(id=119557145489309696, start_id=119557145489309696, end_id=119557145489309696, high_price=951.273010

In [ ]:
from time import sleep
import logging
from pulao.logging import init_logging

from IPython.display import display

from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl
from pulao.constant import *

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(id= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, interval=bar.interval.value, datetime=datetime, turnover=money, open_price=open, close_price=close, high_price=high, low_price=low, volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

sleep(3)

sbar_manager.df_sbar = sbar_manager.df_sbar.with_row_index(name="index")  # offset 可指定起始值
cbar_manager.df_cbar = cbar_manager.df_cbar.with_row_index(name="index")

has_duplicates = sbar_manager.df_sbar.select(pl.col("id").is_duplicated().any()).item()
print("df_sbar has_duplicates:", has_duplicates)
display(sbar_manager.df_sbar)

swing_manager.df_swing

In [ ]:
from pulao.bar import Fractal, CBar

df_fractal_top